In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0', 'google-generativeai>=0.8.0',
    'faker>=24.0.0',
], check=True)

In [ ]:
import os, json, time, threading, uuid, random
from pathlib import Path
from datetime import datetime, timedelta
import yaml, requests
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
DUMMY_ENV_PATH  = WORK_DIR / 'dummy_env.json'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p4a.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

PROFILES_PER_DOMAIN = 1000
DOMAINS             = ['restaurant', 'banking', 'healthcare', 'education']
random.seed(42)

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')
from shared.secrets import load_secrets
SECRETS = load_secrets(require_gemini=False)
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']
with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'tool_registry.yaml') as f: tool_registry = yaml.safe_load(f)
STAGE3_REPO = repos_cfg['repos']['stage3_agent']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage3: {STAGE3_REPO}')

In [ ]:
def rand_pk_phone():
    prefix = random.choice(['0300','0301','0311','0321','0331','0333','0345'])
    return f'{prefix}{random.randint(1000000,9999999)}'

def rand_date_future(days=30):
    d = datetime.now(timezone.utc) + timedelta(days=random.randint(1, days))
    return d.strftime('%Y-%m-%d')

def rand_time():
    h = random.choice([9,10,11,12,14,15,16,17,18])
    m = random.choice([0,15,30,45])
    return f'{h:02d}:{m:02d}'

URDU_NAMES = [
    'احمد علی','محمد حسن','فاطمہ خان','عائشہ رضا','علی رضا',
    'زینب بیگ','عمر فاروق','سارہ احمد','بلال چودھری','حنا ملک',
    'طارق محمود','نادیہ حسین','کامران اختر','روبینہ شاہ','دانیال قریشی',
    'مریم انصاری','شہزاد میر','ثمینہ خان','وقار علی','حسنہ بیگ',
]

def generate_restaurant_profiles(n):
    items = ['چکن کڑاہی','بریانی','نہاری','حلیم','سیخ کباب','دال مخنی','بٹر چکن','پلاؤ','رائتہ','لسی']
    profiles = []
    for i in range(n):
        profiles.append({
            'customer_id':    f'cust_rest_{i+1:04d}',
            'name':           random.choice(URDU_NAMES),
            'phone':          rand_pk_phone(),
            'reservation_id': f'RES{random.randint(10000,99999)}',
            'reservation_date': rand_date_future(14),
            'reservation_time': rand_time(),
            'party_size':     random.randint(2, 10),
            'order_id':       f'ORD{random.randint(100000,999999)}',
            'order_items':    random.sample(items, random.randint(2,4)),
            'order_status':   random.choice(['preparing','on_the_way','delivered','cancelled']),
            'complaint_type': random.choice(['food_quality','late_delivery','wrong_order','service']),
        })
    return profiles

def generate_banking_profiles(n):
    profiles = []
    for i in range(n):
        acc_no = f'PK{random.randint(10000000000000,99999999999999)}'
        profiles.append({
            'customer_id':    f'cust_bank_{i+1:04d}',
            'name':           random.choice(URDU_NAMES),
            'phone':          rand_pk_phone(),
            'account_number': acc_no,
            'account_status': random.choice(['active','blocked','dormant']),
            'balance':        round(random.uniform(500, 500000), 2),
            'card_last4':     str(random.randint(1000,9999)),
            'card_status':    random.choice(['active','blocked']),
            'last_txn_amount': round(random.uniform(100, 50000), 2),
            'block_reason':   random.choice(['suspicious_activity','customer_request','expired',None]),
            'transfer_id':    f'TXN{random.randint(100000000,999999999)}',
        })
    return profiles

def generate_healthcare_profiles(n):
    specialties  = ['جنرل فزیشن','امراض قلب','امراض اطفال','امراض نسواں','ہڈیوں کے امراض','جلدی امراض']
    doctors      = [f'ڈاکٹر {random.choice(URDU_NAMES)}' for _ in range(20)]
    profiles = []
    for i in range(n):
        profiles.append({
            'patient_id':       f'pat_{i+1:04d}',
            'name':             random.choice(URDU_NAMES),
            'phone':            rand_pk_phone(),
            'appointment_id':   f'APT{random.randint(10000,99999)}',
            'doctor_name':      random.choice(doctors),
            'doctor_id':        f'DOC{random.randint(100,999)}',
            'specialty':        random.choice(specialties),
            'appointment_date': rand_date_future(30),
            'appointment_time': rand_time(),
            'appointment_fee':  random.choice([500,800,1000,1500,2000]),
            'prescription_id':  f'RX{random.randint(100000,999999)}',
            'test_type':        random.choice(['CBC','Urine','Blood Sugar','X-Ray','ECG','Ultrasound']),
            'test_result_ready': random.choice([True, False]),
        })
    return profiles

def generate_education_profiles(n):
    programs  = ['بی ایس کمپیوٹر سائنس','بی ایس بزنس','ایف ایس سی','میٹرک','ایم بی اے','بی ایڈ']
    profiles = []
    for i in range(n):
        profiles.append({
            'student_id':     f'STU{random.randint(10000,99999)}',
            'name':           random.choice(URDU_NAMES),
            'phone':          rand_pk_phone(),
            'program':        random.choice(programs),
            'semester':       random.choice(['Spring 2025','Fall 2025','Spring 2026']),
            'cgpa':           round(random.uniform(2.0, 4.0), 2),
            'fee_paid':       random.choice([True, False]),
            'fee_due':        round(random.uniform(15000, 80000), 2),
            'fee_due_date':   rand_date_future(60),
            'application_id': f'APP{random.randint(100000,999999)}',
            'admission_status': random.choice(['shortlisted','waiting_list','rejected','confirmed']),
            'doc_request_id': f'DOC{random.randint(10000,99999)}',
        })
    return profiles

In [ ]:
generators = {
    'restaurant': generate_restaurant_profiles,
    'banking':    generate_banking_profiles,
    'healthcare': generate_healthcare_profiles,
    'education':  generate_education_profiles,
}

dummy_env = {}
for domain, gen_fn in generators.items():
    profiles = gen_fn(PROFILES_PER_DOMAIN)
    dummy_env[domain] = profiles
    print(f'[dummy_env] {domain}: {len(profiles)} profiles generated')

dummy_env['_meta'] = {
    'profiles_per_domain': PROFILES_PER_DOMAIN,
    'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'tool_registry_domains': sorted(set(info.get('domain', '') for info in tool_registry.get('tools', {}).values())),
}

with open(DUMMY_ENV_PATH, 'w', encoding='utf-8') as f:
    json.dump(dummy_env, f, ensure_ascii=False, indent=2)

print(f'\n[p4a] dummy env saved to {DUMMY_ENV_PATH}')
total_profiles = sum(len(v) for k, v in dummy_env.items() if k != '_meta')
print(f'[p4a] total profiles: {total_profiles}')

for attempt in range(8):
    try:
        HF_API.upload_file(
            path_or_fileobj=str(DUMMY_ENV_PATH),
            path_in_repo='dummy_env.json',
            repo_id=STAGE3_REPO,
            repo_type='dataset',
            commit_message=f'dummy_env.json — {total_profiles} customer profiles',
        )
        print(f'[p4a] uploaded to {STAGE3_REPO}')
        break
    except Exception as e:
        time.sleep(min(2**attempt, 60))

print('[done] ready for p4b_generate.ipynb')